In [ ]:
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 데이터셋 안내
### 본 프로젝트는 NASA MOSFET 열화 데이터셋을 사용합니다.
### 용량 관계상 `.mat` 데이터 파일은 포함되어 있지 않으니, 
### 코드를 실행하려면 프로젝트 루트에 `data/` 폴더를 생성하고 데이터 파일을 배치해 주세요.

# 1.파일 불러오기 
### 현재 파일의 확장자가 mat으로 되어있기 때문에 이를 DateFrame으로 변환해주기 위한 함수가 필요



In [ ]:

def load_mat_file(filepath):

    m = scipy.io.loadmat(filepath)
    for k in m.keys(): 
        if not k.startswith("__"):
            main_key = k
    mm = m[main_key][0, 0]
    target_field = 'steadyState'
    data_branch = mm[target_field]
    
    data_list = []
    
    for i in range (data_branch.shape[1]):
        record = data_branch[0, i]
        time_domain = record['timeDomain'][0, 0]
        field = time_domain.dtype.names
        row = {f: time_domain[f].item() for f in field}
        data_list.append(row)
  

    df = pd.DataFrame(data_list)
    df = df.drop(columns='flangeTemperature')
    df['Rds_on'] = df['drainSourceVoltage'] /df['drainCurrent']
    

    return df

### 해당 테스틑 총 42번 진행되어 현 프로젝트에서는 device_num이라고 지정함 이를 불러오기 위해선 다음과 같은 함수가 필요함


In [ ]:
def load_device(device_num, data_dir):
    dfs = []

    run = 1
    while True:
        filepath = f"{data_dir}/Test_{device_num}_run_{run}.mat"
        if not os.path.exists(filepath):
            break
        df = load_mat_file(filepath)
        df['run'] = run
        dfs.append(df)
        run += 1

    return pd.concat(dfs).reset_index(drop=True)

# 그래프 서식 통일을 위한 옵션

In [ ]:
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

BLUE = '#2166AC'
RED = '#D6604D'
GRAY = '#AAAAAA'

# 2.전처리 조건
### 해당 선별된 파일은 모두 5cycles 이상의 파일들 이므로 환경에 따른 열화 조건을 나타낼 수 있다고 판단하여 선별함
### 해당 테스트는 ON-State일 경우에만 측정이 가능하여 통계의 최솟값, 그래프의 모양에 따라 적절한 수치를 반영 
### Id >= 0.5, supplyvoltage >= 3.5 으로 필터링 

In [3]:
devices = [8, 9, 10, 11, 12, 14]
all_dfs = []

fig, axes = plt.subplots(2, 3, figsize=(15, 8))  # 6개 소자 한번에

for idx, d in enumerate(devices):
    df = load_device(d, '../data')

    df_filtered = df[(df['supplyVoltage'] >= 3.5) &
                     (df['drainCurrent'] >= 0.5)
                     ]
    
    
    
    df_filtered['device'] = d
    all_dfs.append(df_filtered)

    ax = axes[idx // 3][idx % 3]
    df_filtered.groupby('run')['Rds_on'].mean().plot(
        ax=ax, marker='o', title=f'Test_{d}'
    )
    ax.set_xlabel('run')
    ax.set_ylabel('Rds_on (Ω)')

plt.tight_layout()
plt.savefig('../results/rds_on_total_trend.png', 
            dpi=150, bbox_inches='tight')
plt.show()  

NameError: name 'plt' is not defined

# 3.Device 별 Rds_on 트렌드 합치기

In [ ]:
all_pivots = []  

for d in devices:
    df = load_device(d, '../data')
    df_filtered = df[(df['supplyVoltage'] >= 3.5) &
                     (df['drainCurrent'] >= 0.5)]
    
    df_filtered = df_filtered.copy()
    df_filtered['device'] = d
    pivot = df_filtered.groupby(['device', 'run'])['Rds_on'].mean().reset_index()
    all_pivots.append(pivot)  

pivot = pd.concat(all_pivots, ignore_index=True) 

# 4.각 Device 별 Rds_on 증가율 표기

In [ ]:
results = []
for device in pivot['device'].unique():
    d = pivot[pivot['device'] == device]

    
    rds_min = d['Rds_on'].min()
    run_min = d.loc[d['Rds_on'].idxmin(), 'run']
    
    
    final_run = d['run'].max()
    
    
    rds_final_median = d[d['run'] == final_run]['Rds_on'].median()
    
    
    increase_rate = (rds_final_median - rds_min) / rds_min * 100
    
    results.append({
        'device': device,
        'rds_bottom': round(rds_min, 4),
        'bottom_run': run_min,
        'final_run': final_run,
        'rds_final_median': round(rds_final_median, 4),
        'increase_rate(%)': round(increase_rate, 1)
    })

result_df = pd.DataFrame(results).sort_values('increase_rate(%)', ascending=False)
print(result_df)

file_name = 'MOSFET_Thermal_Aging_Results.xlsx'

with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    # 시트 이름을 명확하게 부여하여 저장
    result_df.to_excel(writer, sheet_name='Thermal_Degradation_Summary', index=False)

In [ ]:
plt.figure(figsize=(10, 6))

for device in pivot['device'].unique():
    d = pivot[pivot['device'] == device]
    if device == 12:
        plt.plot(d['run'], d['Rds_on'], marker='o', linewidth=2.5, 
                 color='red', label=f'Device {device}')
    else:
        plt.plot(d['run'], d['Rds_on'], marker='o', linewidth=1, 
                 alpha=0.4, color='gray', label=f'Device {device}')

plt.xlabel('Run')
plt.ylabel('Rds_on (Ω)')
plt.title('Rds_on Trend - Device 12 vs Others')
plt.legend()
plt.tight_layout()
plt.savefig('../results/rds_on_trend_with_other', 
            dpi=150, bbox_inches='tight')
plt.show()

# 7. Device 12 심층 분석
### 해당 cycle 중 Rds_on의 수치 분포가 평균적으로 어떻게 변하지는 관찰하기 위한 그래프

In [ ]:
df12 = load_device(12, '../data')
df12_filtered = df12[(df12['supplyVoltage'] >= 3.5) & (df12['drainCurrent'] >= 0.5)]

df12_filtered.boxplot(column='Rds_on', by='run', figsize=(10, 6))
plt.title('Device 12 - Rds_on Distribution by Run')
plt.suptitle('')
plt.xlabel('Run')
plt.ylabel('Rds_on (Ω)')
plt.tight_layout()
plt.savefig('../results/figures/devvice12_rds_on_distribution.png', 
            dpi=150, bbox_inches='tight')
plt.show()

### 정량적 수치를 보기위해 엑셀로 정리

In [ ]:
file_name = 'MOSFET_Thermal_Device12.xlsx'
summary = df12_filtered.describe()

with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    
    summary.to_excel(writer, sheet_name='Thermal_Degradation_Summary', index=False)

### 파라미터 별 상관관계를 전체적으로 보기 위한 그래프

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 10))

columns = ['supplyVoltage', 'packageTemperature', 
           'drainSourceVoltage', 'drainCurrent', 'Rds_on']

for i, col in enumerate(columns):
    axes[i].plot(df12_filtered[col])
    axes[i].set_title(col)
    axes[i].set_xlabel('cycle')

plt.tight_layout()
plt.show()

### Device들의 증가율을 시각적 그래프로 표현 
### Package Temperature와 Rds_on의 상관관계를 나타내기 위한 그래프 
### 전체 Device 별 Temperature와 Rds_on 상관관계를 나타내기 위한 그래프

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = [RED if d == 12 else BLUE for d in result_df['device']]

ax.bar(result_df['device'].astype(str), result_df['increase_rate(%)'],
       color=colors, edgecolor='white', width=0.6)

for i, (val, dev) in enumerate(zip(result_df['increase_rate(%)'], result_df['device'])):
    ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontsize=10)

ax.set_xlabel('Device')
ax.set_ylabel('Rds_on Increase Rate (%)')
ax.set_title('Rds_on Increase Rate by Device (Bottom Run → Run 7)')
plt.tight_layout()
plt.savefig('../results/figures/rds_on_trend.png', 
            dpi=150, bbox_inches='tight')
plt.show()


df12_filtered = df12[(df12['supplyVoltage'] >= 3.5) &
                             (df12['drainCurrent'] >= 0.5)]

pivot_12 = df12_filtered.groupby('run')[['Rds_on', 'packageTemperature']].mean().reset_index()


def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

pivot_12['Rds_norm'] = normalize(pivot_12['Rds_on'])
pivot_12['Temp_norm'] = normalize(pivot_12['packageTemperature'])

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(pivot_12['run'], pivot_12['Rds_norm'],
        marker='o', color=BLUE, linewidth=2, markersize=7, label='Rds_on (normalized)')
ax.plot(pivot_12['run'], pivot_12['Temp_norm'],
        marker='s', color=RED, linewidth=2, markersize=7, linestyle='--', label='Temperature (normalized)')

ax.set_xlabel('Run')
ax.set_ylabel('Normalized Value (0–1)')
ax.set_title('Device 12 - Rds_on vs Temperature (Normalized)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/rds_on_trend.png', 
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
device_dfs = {}

for d in devices:
    df = load_device(d, '../data')
    df_filtered = df[(df['supplyVoltage'] >= 3.5) & 
                     (df['drainCurrent'] >= 0.5)].copy()
    df_filtered['device'] = d
    device_dfs[d] = df_filtered

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()


for idx, device in enumerate(devices):
    
    df_dev = device_dfs[device]
    pivot = df_dev.groupby('run')[['Rds_on', 'packageTemperature']].mean().reset_index()
    
    def normalize(s):
        return (s - s.min()) / (s.max() - s.min())
    
    pivot['Rds_norm'] = normalize(pivot['Rds_on'])
    pivot['Temp_norm'] = normalize(pivot['packageTemperature'])
    
    ax = axes[idx]
    ax.plot(pivot['run'], pivot['Rds_norm'], 
            marker='o', color=BLUE, linewidth=2, markersize=6, label='Rds_on')
    ax.plot(pivot['run'], pivot['Temp_norm'], 
            marker='s', color=RED, linewidth=2, markersize=6, 
            linestyle='--', label='Temperature')
    
    ax.set_title(f'Device {device}')
    ax.set_xlabel('Run')
    ax.set_ylabel('Normalized Value (0–1)')
    ax.legend(fontsize=9)

plt.savefig('../results/figures/temp_rds_on_correlationpng', 
            dpi=150, bbox_inches='tight')
plt.tight_layout()
plt.show()